In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv('/root/.cache/kagglehub/datasets/mohammad2012191/q1-ka-ai-2026/versions/1/Q1_data.csv')
print("Dataset loaded successfully!")
print(f"Dataset shape: {df.shape}\n")

In [ ]:
# Task 2: Write your code here:
print("="*80)
print("TASK 2: First few rows of the dataset")
print("="*80)
print(df.head())
print()

In [ ]:
# Task 3: Write your code here:
print("="*80)
print("TASK 3: Dataset Information")
print("="*80)
df.info()
print()

In [ ]:
# Task 4: Write your code here:
print("="*80)
print("TASK 4: Statistical Description")
print("="*80)
print(df.describe())
print()

In [ ]:
# Task 5: Write your code here:
print("="*80)
print("TASK 5: Target Distribution")
print("="*80)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(df['delivery_time'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
plt.xlabel('Delivery Time (minutes)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of Delivery Time', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(df['delivery_time'], vert=True)
plt.ylabel('Delivery Time (minutes)', fontsize=12)
plt.title('Boxplot of Delivery Time', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
print(f"\nTarget Statistics:")
print(f"Mean: {df['delivery_time'].mean():.2f} minutes")
print(f"Median: {df['delivery_time'].median():.2f} minutes")
print(f"Std Dev: {df['delivery_time'].std():.2f} minutes")
print()

In [ ]:
# Task 1: Write your code here:
print("="*80)
print("PART 2: DATA CLEANING")
print("="*80)
print("\nTask 1: Dropping Order_ID column...")
df_clean = df.drop('Order_ID', axis=1)
print(f"Columns after dropping Order_ID: {df_clean.columns.tolist()}\n")

In [ ]:
# Task 2: Write your code here:
print("Task 2: Handling missing values...")
print("\nMissing values before handling:")
print(df_clean.isnull().sum())
print()

In [ ]:
# Task 3: Write your code here:
missing_pct = print("Task 3: Checking for duplicates...")
duplicates = df_clean.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")
if duplicates > 0:
    df_clean = df_clean.drop_duplicates()
    print(f"Removed {duplicates} duplicate rows")
print(f"Dataset shape after removing duplicates: {df_clean.shape}\n")

In [ ]:
# Task 4: Write your code here:
print("Task 4: Encoding categorical variables...")
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
if 'delivery_time' in categorical_cols:
    categorical_cols.remove('delivery_time')

print(f"Categorical columns found: {categorical_cols}")

if categorical_cols:
    # Using One Hot Encoding
    df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)
    print(f"Applied One-Hot Encoding")
    print(f"Shape after encoding: {df_encoded.shape}")
else:
    df_encoded = df_clean.copy()
    print("No categorical columns to encode")
print()

In [ ]:
# Task 5: Write your code here:
print("Task 5: Applying feature scaling...")
# Separate features and target
X = df_encoded.drop('delivery_time', axis=1)
y = df_encoded['delivery_time']

# Apply StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(f"Applied StandardScaler to all features")
print(f"Feature shape: {X_scaled.shape}")
print(f"Target shape: {y.shape}\n")

In [ ]:
# Task 6: Write your code here:
print("Task 6: Checking target distribution...")
print("Note: This is a regression task, so we check distribution rather than class imbalance")
print(f"Target variable (delivery_time) is continuous: {y.dtype}")
print(f"Range: [{y.min():.2f}, {y.max():.2f}]")
print("The target is NOT imbalanced (this concept applies to classification tasks)")
print()

In [ ]:
# Task 1: Write your code here:
print("="*80)
print("PART 3: MODELING")
print("="*80)

# Task 1: Split dataset (already done above)
print("\nTask 1: Features (X) and Target (y) split completed")
print(f"X shape: {X_scaled.shape}")
print(f"y shape: {y.shape}\n")

In [ ]:
# Task 2,3,4,5: Write your code here:
print("Tasks 2-5: KFold Cross-Validation with RandomForest")
print("-"*80)

# Use KFold (not StratifiedKFold, as this is regression)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
fold_num = 1

for train_idx, val_idx in kfold.split(X_scaled):
    # Split data
    X_train, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train RandomForest model
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)

    # Predict
    y_pred = rf_model.predict(X_val)

    # Evaluate using MAE
    mae = mean_absolute_error(y_val, y_pred)
    mae_scores.append(mae)

    print(f"Fold {fold_num}: MAE = {mae:.4f}")
    fold_num += 1

# Print averaged score
print("-"*80)
print(f"\nAveraged MAE across all folds: {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print()

# Train final model on full data for feature importance and predictions
final_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
final_model.fit(X_scaled, y)
final_predictions = final_model.predict(X_scaled)

In [ ]:
# Task 1: Write your code here:
print("="*80)
print("PART 4: PLOTS")
print("="*80)

# Task 1: Plot feature importance
print("\nTask 1: Plotting feature importance...")
feature_importance = pd.DataFrame({
    'feature': X_scaled.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)
plt.figure(figsize=(12, 8))
plt.barh(range(len(feature_importance)), feature_importance['importance'], color='steelblue')
plt.yticks(range(len(feature_importance)), feature_importance['feature'])
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.title('Feature Importance from RandomForest Model', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nTop 5 Most Important Features:")
print(feature_importance.head())
print()

In [ ]:
# Task 2: Write your code here:
print("Task 2: Plotting predicted delivery time histogram...")
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.hist(final_predictions, bins=30, edgecolor='black', alpha=0.7, color='coral', label='Predicted')
plt.hist(y, bins=30, edgecolor='black', alpha=0.5, color='steelblue', label='Actual')
plt.xlabel('Delivery Time (minutes)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Predicted vs Actual Delivery Time Distribution', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(axis='y', alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(y, final_predictions, alpha=0.5, color='steelblue', s=20)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Delivery Time (minutes)', fontsize=12)
plt.ylabel('Predicted Delivery Time (minutes)', fontsize=12)
plt.title('Actual vs Predicted Delivery Time', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print()

In [ ]:
# Task Bonus: Write your code here:
print("="*80)
print("PART 5: BONUS - ENSEMBLE (RandomForest + CatBoost)")
print("="*80)

try:
    from catboost import CatBoostRegressor

    print("\nTraining ensemble model with KFold cross-validation...")
    print("-"*80)

    ensemble_mae_scores = []
    fold_num = 1

    for train_idx, val_idx in kfold.split(X_scaled):
        # Split data
        X_train, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # Train RandomForest model
        rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        rf_model.fit(X_train, y_train)
        rf_pred = rf_model.predict(X_val)

        # Train CatBoost model
        cb_model = CatBoostRegressor(iterations=100, random_state=42, verbose=0)
        cb_model.fit(X_train, y_train)
        cb_pred = cb_model.predict(X_val)

        # Average predictions
        ensemble_pred = (rf_pred + cb_pred) / 2

        # Calculate MAE on averaged predictions
        mae = mean_absolute_error(y_val, ensemble_pred)
        ensemble_mae_scores.append(mae)

        print(f"Fold {fold_num}: Ensemble MAE = {mae:.4f}")
        fold_num += 1

    print("-"*80)
    print(f"\nAveraged Ensemble MAE across all folds: {np.mean(ensemble_mae_scores):.4f} ± {np.std(ensemble_mae_scores):.4f}")
    print(f"Single RandomForest MAE: {np.mean(mae_scores):.4f}")
    print(f"Improvement: {np.mean(mae_scores) - np.mean(ensemble_mae_scores):.4f}")

except ImportError:
    print("\nCatBoost not installed. To use ensemble, install it with: pip install catboost")
    print("Using only RandomForest for now.")

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)